In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
STREAM_HOST = os.environ.get("SOCKET_HOST", "localhost")
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-structured-streaming-drill").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 17:08:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Drill -- Censor Stream

## Problem

Read a text stream from a socket (localhost:11111) and **filter out** any lines containing  
the word **"shoot"** before writing to the console.

## Setup

Start a netcat server in your terminal:
```bash
# macOS/Linux
nc -lk 11111

# Windows
ncat -lk 11111
```

Type sentences into the terminal. Lines containing "shoot" should be suppressed;  
all other lines should appear in the console output.


In [2]:
bad_word = "shoot"

input_stream = (spark.readStream
    .format("socket")
    .option("host", STREAM_HOST)
    .option("port", 11111)
    .load())

results1 = input_stream # Complete the query here
output1 = results1.writeStream.format("console").outputMode("append")

query = output1.start()
query.awaitTermination(10)
query.stop()


26/06/12 17:08:37 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.
26/06/12 17:08:37 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-05948e9f-adce-414b-8f53-9f98be74e7b8. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/12 17:08:37 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----+
|value|
+-----+
+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+
|               value|
+--------------------+
|shoot this should...|
+--------------------+

-------------------------------------------
Batch: 2
-------------------------------------------
+-----+
|value|
+-----+
|sfdsf|
+-----+

-------------------------------------------
Batch: 3
-------------------------------------------
+------+
| value|
+------+
|sdfsdf|
+------+



26/06/12 17:08:47 WARN DAGScheduler: Failed to cancel job group a3328126-2cb0-4cf5-a435-a96261775bd2. Cannot find active jobs for it.
26/06/12 17:08:47 WARN DAGScheduler: Failed to cancel job group a3328126-2cb0-4cf5-a435-a96261775bd2. Cannot find active jobs for it.


In [3]:
spark.stop()